In [1]:
# CELL 20 — Full build setup: paths, chunk plan, and the v2 generator (identical to CELL 9; only paths, split rule and job list changed)
!pip install -q pikepdf pymupdf pypdf pebble

import os, io, re, glob, json, zlib, shutil, random, logging
import pandas as pd
import pikepdf, pypdf
import pymupdf as fitz
from pypdf import PdfWriter
from pypdf.generic import ArrayObject, StreamObject, NameObject
from PIL import Image, ImageDraw, ImageFont
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("pikepdf").setLevel(logging.ERROR)

# ---------- paths ----------
hits = sorted(glob.glob("/kaggle/input/**/clean_pool.csv", recursive=True))
assert hits, "clean_pool.csv not found - is PDP BPTP attached as input?"
CLEAN_POOL = hits[0]
SRC = os.path.join(os.path.dirname(CLEAN_POOL), "govdocs_pdfs")
assert os.path.isdir(SRC), f"govdocs_pdfs not found next to {CLEAN_POOL}"
WORK = "/kaggle/working/full"
OUT = f"{WORK}/tmp_pdfs"      # PDFs of the current chunk only (deleted after extraction)
META = f"{WORK}/meta"         # one small JSON per original (manifest rows)
CHUNKS = f"{WORK}/chunks"     # per-chunk results: features CSV + graphs file
for d in (OUT, META, CHUNKS):
    os.makedirs(d, exist_ok=True)

TIME_LIMIT = 180
CHUNK_SIZE = 300
WORKERS = 4
FAMILIES = ["acrobat distiller windows", "acrobat distiller for", "acrobat pdfwriter for",
            "adobe pdf library", "corel pdf engine"]
TOOLS = ["pikepdf", "pymupdf", "pypdf"]

LIT = re.compile(rb"\((?:[^()\\]|\\.)*\)", re.S)
TAIL = re.compile(rb"(?:\s+|-?(?:\d+\.?\d*|\.\d+)(?=[\s(\]])|\((?:[^()\\]|\\.)*\))*\s*(?:\]\s*TJ|Tj|'|\")")
VALUE = re.compile(rb"\d[\d,\.]*\d")

def make_stamp(text):
    img = Image.new("RGBA", (420, 140), (255, 255, 255, 0))
    dr = ImageDraw.Draw(img)
    dr.rectangle([4, 4, 415, 135], outline=(200, 0, 0, 255), width=8)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 60)
    except Exception:
        font = ImageFont.load_default()
    dr.text((24, 34), text, fill=(200, 0, 0, 255), font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

PNG_HONEST = make_stamp("RECEIVED")
PNG_FAKE = make_stamp("PAID")

def digit_positions(seg):
    pos, i, n = [], 1, len(seg) - 1
    while i < n:
        b = seg[i]
        if b == 0x5C:
            j = i + 1
            if j < n and 48 <= seg[j] <= 55:
                k = j
                while k < n and k < j + 3 and 48 <= seg[k] <= 55:
                    k += 1
                i = k
            else:
                i = j + 1
            continue
        if 48 <= b <= 57:
            pos.append(i)
        i += 1
    return pos

def candidates(data):
    digits, values = [], []
    for m in LIT.finditer(data):
        if not TAIL.match(data[m.end():m.end() + 300]):
            continue
        seg = m.group()
        digits += [m.start() + p for p in digit_positions(seg)]
        for v in VALUE.finditer(seg):
            if v.start() > 0 and seg[v.start() - 1] == 0x5C:
                continue
            if sum(c in b"0123456789" for c in v.group()) >= 3:
                values.append((m.start() + v.start(), m.start() + v.end()))
    return digits, values

def new_value(old, rng):
    s = old.decode("latin-1")
    two_dec = s.count(".") == 1 and len(s.split(".")[1]) == 2
    intpart = s.split(".")[0] if two_dec else s
    nd = max(1, sum(c.isdigit() for c in intpart) + rng.choice([-1, 1, 2]))
    n = rng.randint(10 ** (nd - 1), 10 ** nd - 1)
    out = (f"{n:,}" if "," in s else str(n)) + (f".{rng.randint(0, 99):02d}" if two_dec else "")
    if out == s:
        out = str(n + 1)
    return out.encode()

def pk_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def choose_specs(src, rng):
    dspec = vspec = None
    with pikepdf.open(src) as p:
        order = list(range(len(p.pages)))
        rng.shuffle(order)
        for i in order:
            for k, s in enumerate(pk_streams(p, i)):
                try:
                    data = s.read_bytes()
                except Exception:
                    continue
                d, v = candidates(data)
                if dspec is None and d:
                    pos = rng.choice(d)
                    old = data[pos:pos + 1]
                    new = bytes([rng.choice([c for c in b"0123456789" if c != old[0]])])
                    dspec = dict(page=i, k=k, start=pos, end=pos + 1, old=old, new=new)
                if vspec is None and v:
                    st, en = rng.choice(v)
                    vspec = dict(page=i, k=k, start=st, end=en, old=data[st:en], new=new_value(data[st:en], rng))
                if dspec and vspec:
                    return dspec, vspec
    return dspec, vspec

def apply_spec(data, spec):
    if data[spec["start"]:spec["end"]] != spec["old"]:
        raise RuntimeError("edit location mismatch")
    return data[:spec["start"]] + spec["new"] + data[spec["end"]:]

def edit_pikepdf(pdf, spec):
    s = pk_streams(pdf, spec["page"])[spec["k"]]
    s.write(apply_spec(s.read_bytes(), spec))

def edit_fitz(doc, spec):
    x = doc[spec["page"]].get_contents()[spec["k"]]
    doc.update_stream(x, apply_spec(doc.xref_stream(x), spec))

def py_streams(page):
    c = page.get("/Contents")
    if c is None:
        return []
    c = c.get_object()
    out = []
    for it in (list(c) if isinstance(c, ArrayObject) else [c]):
        o = it.get_object()
        if isinstance(o, StreamObject):
            out.append(o)
    return out

def edit_pypdf(w, spec):
    o = py_streams(w.pages[spec["page"]])[spec["k"]]
    new = apply_spec(o.get_data(), spec)
    flt = o.get("/Filter")
    if flt is not None and flt != "/FlateDecode" and list(flt) != ["/FlateDecode"]:
        o[NameObject("/Filter")] = NameObject("/FlateDecode")
    if "/DecodeParms" in o:
        del o["/DecodeParms"]
    o.set_data(new)

def run_tool(tool, src, out, spec, mode):
    if tool == "pikepdf":
        with pikepdf.open(src) as p:
            if spec:
                edit_pikepdf(p, spec)
            if mode == "recompress":
                p.save(out, compress_streams=True, recompress_flate=True,
                       object_stream_mode=pikepdf.ObjectStreamMode.generate)
            elif mode == "linearize":
                p.save(out, linearize=True)
            else:
                p.save(out)
    elif tool == "pymupdf":
        d = fitz.open(src)
        if spec:
            edit_fitz(d, spec)
        if mode == "recompress":
            for pg in d:
                for x in pg.get_contents():
                    d.update_stream(x, d.xref_stream(x))
            d.save(out, garbage=4, deflate=True)
        else:
            d.save(out)
        d.close()
    else:
        w = PdfWriter(clone_from=src)
        if spec:
            edit_pypdf(w, spec)
        if mode == "recompress":
            for pg in w.pages:
                pg.compress_content_streams()
        with open(out, "wb") as f:
            w.write(f)

def fitz_incr(src, out, action):
    shutil.copy(src, out)
    d = fitz.open(out)
    try:
        if d.is_repaired:
            raise RuntimeError("file needed repair")
        res = action(d)
        d.saveIncr()
        return res
    finally:
        d.close()

def fitz_full(src, out, action):
    d = fitz.open(src)
    try:
        res = action(d)
        d.save(out)
        return res
    finally:
        d.close()

def add_note(d, i):
    r = d[i].rect
    d[i].add_text_annot(fitz.Point(r.x0 + 36, r.y0 + 36), "Reviewed")

def add_stamp(d, i, png):
    r = d[i].rect
    d[i].insert_image(fitz.Rect(r.x1 - 210, r.y0 + 40, r.x1 - 40, r.y0 + 97), stream=png, overlay=True)

def add_textbox(d, i, text):
    r = d[i].rect
    d[i].add_freetext_annot(fitz.Rect(r.x0 + 40, r.y1 - 80, r.x0 + 300, r.y1 - 58), text, fontsize=11)

def cover_replace(d, rng):
    order = list(range(len(d)))
    rng.shuffle(order)
    for i in order:
        words = [w for w in d[i].get_text("words") if re.search(r"\d", w[4]) and len(w[4]) >= 2]
        if not words:
            continue
        x0, y0, x1, y1, txt = rng.choice(words)[:5]
        r = fitz.Rect(x0, y0, x1, y1)
        if r.is_empty or r.height < 3:
            continue
        new = "".join(rng.choice("0123456789") if c.isdigit() else c for c in txt)
        if new == txt:
            new = txt[:-1] + ("7" if txt[-1] != "7" else "3")
        d[i].draw_rect(r, color=None, fill=(1, 1, 1), overlay=True)
        d[i].insert_text(fitz.Point(x0, y1 - r.height * 0.2), new, fontsize=max(5, r.height * 0.8),
                         fontname="helv", color=(0, 0, 0))
        return {"page": i, "old": txt, "new": new}
    raise RuntimeError("no numeric word for cover-and-replace")

def cleanup(stem, keep_meta=False):
    for p in glob.glob(os.path.join(OUT, f"{stem}__*.pdf")):
        os.remove(p)
    mp = os.path.join(META, stem + ".json")
    if not keep_meta and os.path.exists(mp):
        os.remove(mp)

def split_of(name):
    return "dev" if zlib.crc32(name.encode()) % 100 < 15 else "train"

# ---------- one original -> 15 versions (same as CELL 9; only the split line differs) ----------
def process(job):
    name, fam = job
    fitz.TOOLS.mupdf_display_errors(False)
    stem = name[:-4]
    src = os.path.join(SRC, name)
    meta_path = os.path.join(META, stem + ".json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            rows = json.load(fh)
        if all(os.path.exists(os.path.join(OUT, r["out_file"])) for r in rows):
            return name, rows, None
    cleanup(stem)
    try:
        rng = random.Random(zlib.crc32(name.encode()) + 2026)
        split = split_of(name)
        t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
        dspec, vspec = choose_specs(src, rng)
        if dspec is None or vspec is None:
            raise RuntimeError("no editable number")
        P = lambda v: os.path.join(OUT, f"{stem}__{v}.pdf")
        pg = dspec["page"]

        shutil.copy(src, P("G0_original"))
        run_tool(t_main, src, P("G1_resave"), None, "resave")
        run_tool(t_comp, src, P("G2_recompress"), None, "recompress")
        fitz_incr(src, P("G3_note_incr"), lambda d: add_note(d, pg))
        run_tool("pikepdf", src, P("G4_linearize"), None, "linearize")
        fitz_full(src, P("G5_stamp_honest"), lambda d: add_stamp(d, pg, PNG_HONEST))
        fitz_incr(src, P("G6_textbox_honest"), lambda d: add_textbox(d, pg, "Approved - HR dept."))

        run_tool(t_main, src, P("F1_digit_resave"), dspec, "resave")
        run_tool(t_comp, src, P("F2_digit_recompress"), dspec, "recompress")
        fitz_incr(src, P("F3_digit_incr"), lambda d: edit_fitz(d, dspec))
        run_tool(t_main, src, P("F4_value_resave"), vspec, "resave")
        cov = fitz_full(src, P("F5_cover_replace"), lambda d: cover_replace(d, rng))
        run_tool("pikepdf", src, P("F6_digit_linearize"), dspec, "linearize")
        fitz_full(src, P("F7_stamp_fake"), lambda d: add_stamp(d, pg, PNG_FAKE))
        fitz_incr(src, P("F8_textbox_fake"), lambda d: add_textbox(d, pg, "TOTAL DUE: $98,450.00"))

        dold, dnew = dspec["old"].decode("latin-1"), dspec["new"].decode("latin-1")
        vold, vnew = vspec["old"].decode("latin-1"), vspec["new"].decode("latin-1")
        spec = [
            ("G0_original", 0, "none", "original", "none", None, None, None, "G0"),
            ("G1_resave", 0, t_main, "full", "none", None, None, None, "G1"),
            ("G2_recompress", 0, t_comp, "full_recompress", "none", None, None, None, "G2"),
            ("G3_note_incr", 0, "pymupdf", "incremental", "note", pg, None, None, "G3"),
            ("G4_linearize", 0, "pikepdf", "linearize", "none", None, None, None, "G4"),
            ("G5_stamp_honest", 0, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("G6_textbox_honest", 0, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
            ("F1_digit_resave", 1, t_main, "full", "digit", pg, dold, dnew, "G1"),
            ("F2_digit_recompress", 1, t_comp, "full_recompress", "digit", pg, dold, dnew, "G2"),
            ("F3_digit_incr", 1, "pymupdf", "incremental", "digit", pg, dold, dnew, "G3"),
            ("F4_value_resave", 1, t_main, "full", "value", vspec["page"], vold, vnew, "G1"),
            ("F5_cover_replace", 1, "pymupdf", "full", "cover", cov["page"], cov["old"], cov["new"], "G1"),
            ("F6_digit_linearize", 1, "pikepdf", "linearize", "digit", pg, dold, dnew, "G4"),
            ("F7_stamp_fake", 1, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("F8_textbox_fake", 1, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
        ]
        rows = [{"out_file": f"{stem}__{v}.pdf", "orig_file": name, "family": fam, "variant": v,
                 "label": lab, "tool": tool, "save_mode": mode, "edit_type": et, "edit_page": ep,
                 "old_value": ov, "new_value": nv, "twin": tw, "split": split}
                for v, lab, tool, mode, et, ep, ov, nv, tw in spec]
        with open(meta_path, "w") as fh:
            json.dump(rows, fh)
        return name, rows, None
    except Exception as ex:
        cleanup(stem)
        return name, None, f"{type(ex).__name__}: {str(ex)[:50]}"

# ---------- job list and chunk plan ----------
pool = pd.read_csv(CLEAN_POOL)
pool = pool[pool.producer_family.isin(FAMILIES)].copy()
pool["fam_order"] = pool.producer_family.map({f: i for i, f in enumerate(FAMILIES)})
pool = pool.sort_values(["fam_order", "file"])
JOBS = list(zip(pool.file, pool.producer_family))
CHUNK_LIST = [JOBS[i:i + CHUNK_SIZE] for i in range(0, len(JOBS), CHUNK_SIZE)]

print("Originals dir:", SRC, "|", len(glob.glob(SRC + "/*.pdf")), "PDFs")
print("\n=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===")
print(pool.producer_family.value_counts().reindex(FAMILIES).to_string())
print("Total originals:", len(JOBS), "| expected files if all succeed:", len(JOBS) * 15)
print("Dev share (originals):", round(sum(split_of(n) == "dev" for n, _ in JOBS) / len(JOBS) * 100, 1), "%")
print("\nChunks:", len(CHUNK_LIST), "of up to", CHUNK_SIZE, "originals")
print("Chunks already finished:", len(glob.glob(f"{CHUNKS}/chunk_*_features.csv")))
print("\nLibrary versions -> pikepdf", pikepdf.__version__, "| pymupdf", fitz.VersionBind, "| pypdf", pypdf.__version__)
free = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk in /kaggle/working: {free:.1f} GB")

Originals dir: /kaggle/input/notebooks/atharvagandhi/pdp-bptp/govdocs_pdfs | 5029 PDFs

=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===
producer_family
acrobat distiller windows    1566
acrobat distiller for         659
acrobat pdfwriter for         428
adobe pdf library             153
corel pdf engine              108
Total originals: 2914 | expected files if all succeed: 43710
Dev share (originals): 14.8 %

Chunks: 10 of up to 300 originals
Chunks already finished: 0

Library versions -> pikepdf 10.17.0 | pymupdf 2.0 | pypdf 6.21.2
Free disk in /kaggle/working: 20.9 GB


In [3]:
# CELL 21 — Feature extractor (exact CELL 10 code) + PDF->graph converter (exact fixed CELL 12 code), path-based; quick self-test
import os, re, zlib, math, gzip, pickle, logging, collections
import numpy as np
import pandas as pd
import pikepdf

logging.getLogger("pikepdf").setLevel(logging.ERROR)

FEAT_TIME_LIMIT = 60
GRAPH_TIME_LIMIT = 90
MAX_STREAMS = 120
TOOL_WORDS = [b"pypdf", b"pymupdf", b"mupdf", b"pikepdf", b"qpdf"]
STREAM_BODY = re.compile(rb"(?<!end)stream\r?\n.*?endstream", re.S)
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}

def ratio(a, b):
    return a / b if b else 0.0

# =====================================================================================
# PART A — document-level features (identical to CELL 10)
# =====================================================================================
def file_features(raw):
    f = {}
    skel = STREAM_BODY.sub(b"stream endstream", raw)
    f["size_log"] = math.log10(len(raw) + 1)
    m = re.match(rb"\s*%PDF-(\d)\.(\d)", raw[:64])
    f["version"] = int(m.group(1)) + int(m.group(2)) / 10 if m else 0
    f["binary_comment"] = int(bool(re.match(rb"\s*%PDF-\d\.\d\s*[\r\n]+%[\x80-\xff]{4}", raw[:40])))
    f["n_eof"] = raw.count(b"%%EOF")
    f["n_startxref"] = raw.count(b"startxref")
    f["xref_table"] = int(bool(re.search(rb"(^|[\r\n])xref\s*[\r\n]", skel)))
    f["xref_stream"] = int(bool(re.search(rb"/Type\s*/XRef", skel)))
    f["obj_streams"] = int(bool(re.search(rb"/Type\s*/ObjStm", skel)))
    f["linearized"] = int(b"/Linearized" in raw[:2048])
    f["true_revisions"] = f["n_eof"] - f["linearized"]
    crlf = skel.count(b"\r\n")
    cr = skel.count(b"\r") - crlf
    lf = skel.count(b"\n") - crlf
    tot = crlf + cr + lf
    f["eol_crlf"], f["eol_cr"], f["eol_lf"] = ratio(crlf, tot), ratio(cr, tot), ratio(lf, tot)
    objs = re.findall(rb"(\d+)\s+(\d+)\s+obj", skel)
    nums = [int(a) for a, _ in objs]
    f["n_obj_defs"] = len(nums)
    f["obj_order_increasing"] = ratio(sum(1 for a, b in zip(nums, nums[1:]) if b > a), max(len(nums) - 1, 1))
    f["obj_gen_nonzero"] = ratio(sum(1 for _, g in objs if g != b"0"), len(objs))
    f["obj_max_over_count"] = ratio(max(nums) if nums else 0, len(set(nums)))
    f["obj_redefined"] = len(nums) - len(set(nums))
    f["obj_newline_dict"] = ratio(len(re.findall(rb"obj\s*[\r\n]+\s*<<", skel)), len(nums))
    f["dict_open_space"] = ratio(skel.count(b"<< /"), skel.count(b"<<"))
    f["dict_close_space"] = ratio(skel.count(b" >>"), skel.count(b">>"))
    nstream = len(re.findall(rb"(?<!end)stream", skel))
    f["stream_kw_crlf"] = ratio(len(re.findall(rb"(?<!end)stream\r\n", raw)), nstream)
    f["endobj_newline_before"] = ratio(len(re.findall(rb"[\r\n]endobj", skel)), skel.count(b"endobj"))
    f["reals_dot_zero"] = ratio(len(re.findall(rb"\s\d+\.0[\s/\]>]", skel)), len(re.findall(rb"\s\d+\.\d+[\s/\]>]", skel)))
    f["has_ID"] = int(b"/ID" in skel)
    f["tool_word_in_file"] = int(any(w in skel.lower() for w in TOOL_WORDS))
    return f

def stream_sig(s):
    d = {}
    flt = s.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    d["filter"] = "+".join(names) if names else "none"
    d["has_parms"] = int(s.get("/DecodeParms") is not None)
    d["len_indirect"] = int(bool(getattr(s.get("/Length"), "is_indirect", False)))
    d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["ratio"] = -1, 0, 0, 0, 0, 0.0
    raw = s.read_raw_bytes()
    if names == ["/FlateDecode"] and len(raw) >= 6:
        d["hdr"] = raw[0] * 256 + raw[1]
        try:
            dob = zlib.decompressobj()
            dec = dob.decompress(raw)
            body = raw[:len(raw) - len(dob.unused_data)]
            d["tail_junk"] = int(len(dob.unused_data) > 0)
            d["ratio"] = ratio(len(body), len(dec))
            d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
            d["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
            if len(dec) <= 2_000_000:
                for lv in range(1, 10):
                    if zlib.compress(dec, lv) == body:
                        d["zlevel"] = lv
                        break
        except Exception:
            d["hdr"] = -2
    return d

def resource_features(pdf):
    f = {"l5_n_annots": 0, "l5_n_freetext": 0, "l5_n_text_notes": 0, "l5_n_img_xobj": 0, "l5_n_fonts": 0,
         "l5_n_std14_unembedded": 0, "l5_frac_pages_multi_stream": 0.0, "l5_max_streams_per_page": 0}
    fonts_seen, imgs_seen, multi = set(), set(), 0
    for page in pdf.pages:
        po = page.obj
        annots = po.get("/Annots")
        if isinstance(annots, pikepdf.Array):
            for a in annots:
                try:
                    st = str(a.get("/Subtype", ""))
                except Exception:
                    continue
                f["l5_n_annots"] += 1
                f["l5_n_freetext"] += int(st == "/FreeText")
                f["l5_n_text_notes"] += int(st == "/Text")
        c = po.get("/Contents")
        ns = len(c) if isinstance(c, pikepdf.Array) else (1 if c is not None else 0)
        multi += int(ns > 1)
        f["l5_max_streams_per_page"] = max(f["l5_max_streams_per_page"], ns)
        try:
            res = page.resources
        except Exception:
            res = po.get("/Resources")
        if res is None:
            continue
        fdict = res.get("/Font")
        if isinstance(fdict, pikepdf.Dictionary):
            for _, fo in fdict.items():
                key = getattr(fo, "objgen", (0, 0))
                key = key if key != (0, 0) else id(fo)
                if key in fonts_seen:
                    continue
                fonts_seen.add(key)
                base = str(fo.get("/BaseFont", "")).lstrip("/")
                desc = fo.get("/FontDescriptor")
                embedded = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                if base in STD14 and not embedded and str(fo.get("/Subtype", "")) != "/Type0":
                    f["l5_n_std14_unembedded"] += 1
        xdict = res.get("/XObject")
        if isinstance(xdict, pikepdf.Dictionary):
            for _, xo in xdict.items():
                key = getattr(xo, "objgen", (0, 0))
                if key in imgs_seen:
                    continue
                try:
                    if str(xo.get("/Subtype", "")) == "/Image":
                        imgs_seen.add(key)
                except Exception:
                    pass
    f["l5_n_fonts"] = len(fonts_seen)
    f["l5_n_img_xobj"] = len(imgs_seen)
    f["l5_frac_pages_multi_stream"] = ratio(multi, len(pdf.pages))
    return f

def extract_features(path):
    with open(path, "rb") as fh:
        raw = fh.read()
    f = {"out_file": os.path.basename(path)}
    f.update(file_features(raw))
    with pikepdf.open(path) as pdf:
        info = pdf.docinfo
        prod = str(info.get("/Producer", "")) if info is not None else ""
        f["has_producer"] = int(bool(prod.strip()))
        f["producer_len"] = len(prod)
        cd, md = str(info.get("/CreationDate", "")), str(info.get("/ModDate", ""))
        f["moddate_eq_creation"] = int(bool(cd) and cd == md)
        f["has_xmp"] = int("/Metadata" in pdf.Root)
        f["n_pages"] = len(pdf.pages)
        try:
            f.update(resource_features(pdf))
        except Exception:
            pass
        streams = []
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream):
                    streams.append((pi, s))
            if len(streams) >= MAX_STREAMS:
                break
        streams = streams[:MAX_STREAMS]
        sigs = []
        for pi, s in streams:
            try:
                sigs.append((pi, getattr(s, "objgen", (-1, 0))[0], stream_sig(s)))
            except Exception:
                pass
        cs_nums = {num for _, num, _ in sigs}

    n = len(sigs)
    f["cs_n"] = n
    if n:
        fl = [d["filter"] for _, _, d in sigs]
        f["cs_frac_flate"] = ratio(fl.count("/FlateDecode"), n)
        f["cs_frac_lzw"] = ratio(sum("LZW" in x for x in fl), n)
        f["cs_frac_none"] = ratio(fl.count("none"), n)
        f["cs_n_filters"] = len(set(fl))
        flate = [d for _, _, d in sigs if d["filter"] == "/FlateDecode"]
        f["cs_frac_zlib_repro"] = ratio(sum(d["zlevel"] > 0 for d in flate), len(flate)) if flate else np.nan
        f["cs_n_hdr"] = len({d["hdr"] for _, _, d in sigs})
        f["cs_n_zlevel"] = len({d["zlevel"] for _, _, d in sigs})
        f["cs_frac_sync"] = np.mean([d["sync_end"] for _, _, d in sigs])
        f["cs_frac_adler"] = np.mean([d["adler_ok"] for _, _, d in sigs])
        f["cs_frac_junk"] = np.mean([d["tail_junk"] for _, _, d in sigs])
        f["cs_frac_len_indirect"] = np.mean([d["len_indirect"] for _, _, d in sigs])
        r = np.array([d["ratio"] for _, _, d in sigs])
        f["cs_ratio_std"] = float(r.std())
        f["cs_ratio_max_z"] = float(np.max(np.abs(r - r.mean())) / r.std()) if r.std() > 0 else 0.0
        keys = [(d["filter"], d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["has_parms"], d["len_indirect"]) for _, _, d in sigs]
        cnt = collections.Counter(keys)
        f["cs_n_sig"] = len(cnt)
        f["cs_majority_share"] = max(cnt.values()) / n
        f["cs_n_singletons"] = sum(1 for v in cnt.values() if v == 1) if n >= 3 else 0
        odd = [pi for (pi, _, _), k in zip(sigs, keys) if cnt[k] == 1] if n >= 3 else []
        f["loc_page"] = odd[0] if len(odd) == 1 else -1

    f.update({"upd_n_objs": 0, "upd_touch_content": 0, "upd_has_annot": 0, "upd_has_page": 0, "upd_has_sig": 0})
    if f["true_revisions"] > 1:
        ends = [m.end() for m in re.finditer(rb"%%EOF", raw)]
        tail = STREAM_BODY.sub(b"stream endstream", raw[ends[-2]:])
        objs = re.findall(rb"(\d+)\s+\d+\s+obj(.*?)endobj", tail, re.S)
        f["upd_n_objs"] = len(objs)
        for num, body in objs:
            if int(num) in cs_nums:
                f["upd_touch_content"] = 1
            if re.search(rb"/Type\s*/Annot|/Subtype\s*/(Text|FreeText)\b", body):
                f["upd_has_annot"] = 1
            if re.search(rb"/Type\s*/Page\b", body):
                f["upd_has_page"] = 1
            if b"/ByteRange" in body:
                f["upd_has_sig"] = 1
    return f

# =====================================================================================
# PART B — PDF -> graph (identical to fixed CELL 12)
# =====================================================================================
NODE_TYPES = ["catalog", "pages", "page", "content", "font", "fontdesc", "fontfile", "image", "form", "annot",
              "metadata", "xref", "extgstate", "other_dict", "array", "other_stream", "other"]
EDGE_KEYS = ["/Parent", "/Kids", "/Contents", "/Resources", "/Font", "/XObject", "/Annots", "/FontDescriptor",
             "/FontFile", "/Metadata", "/Pages", "other"]
HDRS = [0x789C, 0x78DA, 0x7801, 0x785E]
FEAT_NAMES = (["t_" + t for t in NODE_TYPES] + ["is_stream", "raw_len_log", "n_keys_log"] +
              ["f_flate", "f_lzw", "f_dct", "f_none", "f_other"] +
              ["h_789C", "h_78DA", "h_7801", "h_785E", "h_other"] +
              ["zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio",
               "objnum_rank", "gen_nonzero", "std14_unembedded", "annot_freetext", "annot_text",
               "len_indirect", "entropy"])
D = len(FEAT_NAMES)
LEN_REF = re.compile(rb"/Length\s+\d+\s+\d+\s+R")

def edge_key(k):
    if k is None:
        return "other"
    if k.startswith("/FontFile"):
        return "/FontFile"
    return k if k in EDGE_KEYS else "other"

def collect_refs(o, carried, out, depth=0):
    if depth > 8:
        return
    try:
        if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
            for k, v in o.items():
                kk = k if (k in EDGE_KEYS or k.startswith("/FontFile")) else carried
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(kk), v.objgen))
                else:
                    collect_refs(v, kk, out, depth + 1)
        elif isinstance(o, pikepdf.Array):
            for v in o:
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(carried), v.objgen))
                else:
                    collect_refs(v, carried, out, depth + 1)
    except Exception:
        pass

def node_type(o, content_set):
    if isinstance(o, pikepdf.Stream):
        if o.objgen in content_set:
            return "content"
        st, ty = str(o.get("/Subtype", "")), str(o.get("/Type", ""))
        if st == "/Image":
            return "image"
        if st == "/Form":
            return "form"
        if ty == "/Metadata":
            return "metadata"
        if ty in ("/XRef", "/ObjStm"):
            return "xref"
        if "/Length1" in o or st in ("/Type1C", "/CIDFontType0C", "/OpenType"):
            return "fontfile"
        return "other_stream"
    if isinstance(o, pikepdf.Dictionary):
        ty, st = str(o.get("/Type", "")), str(o.get("/Subtype", ""))
        mapping = {"/Catalog": "catalog", "/Pages": "pages", "/Page": "page", "/Font": "font",
                   "/FontDescriptor": "fontdesc", "/Annot": "annot", "/ExtGState": "extgstate"}
        if ty in mapping:
            return mapping[ty]
        if st in ("/Text", "/FreeText", "/Link", "/Widget", "/Stamp", "/Square", "/Highlight"):
            return "annot"
        return "other_dict"
    if isinstance(o, pikepdf.Array):
        return "array"
    return "other"

def shannon(raw):
    if not raw:
        return 0.0
    b = np.frombuffer(raw[:2_000_000], dtype=np.uint8)
    c = np.bincount(b, minlength=256)
    p = c[c > 0] / len(b)
    return float(-(p * np.log2(p)).sum() / 8)

def stream_feats(o, is_content):
    v = {"raw_len_log": 0.0, "f": "f_none", "h": None, "zlevel": 0, "zlib_repro": 0, "adler_ok": 0,
         "sync_end": 0, "tail_junk": 0, "ratio": 0.0, "len_indirect": 0, "entropy": 0.0}
    try:
        v["len_indirect"] = int(bool(LEN_REF.search(o.stream_dict.unparse())))
    except Exception:
        pass
    flt = o.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    raw = o.read_raw_bytes()
    v["raw_len_log"] = math.log10(len(raw) + 1) / 7
    v["entropy"] = shannon(raw)
    if not names:
        v["f"] = "f_none"
    elif names == ["/FlateDecode"]:
        v["f"] = "f_flate"
    elif any("LZW" in n for n in names):
        v["f"] = "f_lzw"
    elif any("DCT" in n for n in names):
        v["f"] = "f_dct"
    else:
        v["f"] = "f_other"
    if v["f"] == "f_flate" and len(raw) >= 6:
        hdr = raw[0] * 256 + raw[1]
        v["h"] = "h_%04X" % hdr if hdr in HDRS else "h_other"
        if is_content or len(raw) <= 200_000:
            try:
                dob = zlib.decompressobj()
                dec = dob.decompress(raw)
                body = raw[:len(raw) - len(dob.unused_data)]
                v["tail_junk"] = int(len(dob.unused_data) > 0)
                v["ratio"] = min(len(body) / max(len(dec), 1), 1.5)
                v["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
                v["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
                if len(dec) <= 2_000_000:
                    for lv in range(1, 10):
                        if zlib.compress(dec, lv) == body:
                            v["zlevel"], v["zlib_repro"] = lv, 1
                            break
            except Exception:
                pass
    return v

def to_graph(path):
    with pikepdf.open(path) as pdf:
        content_page = {}
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream) and s.objgen not in content_page:
                    content_page[s.objgen] = pi
        objs = [o for o in pdf.objects if hasattr(o, "objgen") and o.objgen != (0, 0)]
        idx = {o.objgen: i for i, o in enumerate(objs)}
        max_num = max([g[0] for g in idx] + [1])
        X = np.zeros((len(objs), D), dtype=np.float16)
        node_page = np.full(len(objs), -1, dtype=np.int16)
        col = {n: j for j, n in enumerate(FEAT_NAMES)}
        src, dst, et = [], [], []
        for i, o in enumerate(objs):
            t = node_type(o, content_page)
            X[i, col["t_" + t]] = 1
            X[i, col["objnum_rank"]] = o.objgen[0] / max_num
            X[i, col["gen_nonzero"]] = int(o.objgen[1] != 0)
            if o.objgen in content_page:
                node_page[i] = content_page[o.objgen]
            try:
                if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
                    X[i, col["n_keys_log"]] = math.log10(len(list(o.keys())) + 1)
                if isinstance(o, pikepdf.Stream):
                    X[i, col["is_stream"]] = 1
                    sf = stream_feats(o, t == "content")
                    X[i, col["raw_len_log"]] = sf["raw_len_log"]
                    X[i, col[sf["f"]]] = 1
                    if sf["h"]:
                        X[i, col[sf["h"]]] = 1
                    X[i, col["zlevel"]] = sf["zlevel"] / 9
                    for k in ["zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy"]:
                        X[i, col[k]] = sf[k]
                if t == "font":
                    base = str(o.get("/BaseFont", "")).lstrip("/")
                    desc = o.get("/FontDescriptor")
                    emb = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                    X[i, col["std14_unembedded"]] = int(base in STD14 and not emb and str(o.get("/Subtype", "")) != "/Type0")
                if t == "annot":
                    st = str(o.get("/Subtype", ""))
                    X[i, col["annot_freetext"]] = int(st == "/FreeText")
                    X[i, col["annot_text"]] = int(st == "/Text")
            except Exception:
                pass
            refs = []
            collect_refs(o, None, refs)
            for k, tgt in refs:
                if tgt in idx:
                    src.append(i)
                    dst.append(idx[tgt])
                    et.append(EDGE_KEYS.index(k))
    return {"out_file": os.path.basename(path), "x": X, "edge_index": np.array([src, dst], dtype=np.int32),
            "edge_type": np.array(et, dtype=np.int8), "node_page": node_page}

# =====================================================================================
# Quick self-test on 3 original PDFs (reads only, writes nothing)
# =====================================================================================
for name, fam in [JOBS[0], JOBS[len(JOBS) // 2], JOBS[-1]]:
    p = os.path.join(SRC, name)
    fe = extract_features(p)
    gr = to_graph(p)
    print(f"{name} ({fam}): {len(fe) - 1} feature values | graph {gr['x'].shape[0]} nodes, "
          f"{gr['edge_index'].shape[1]} edges, {gr['x'].shape[1]} node features")
print("\nCELL 21 ready. Expected: 62 feature values (61 if a file has no page streams), 43 node features.")

000010.pdf (acrobat distiller windows): 62 feature values | graph 45 nodes, 60 edges, 43 node features
018877.pdf (acrobat distiller windows): 62 feature values | graph 183 nodes, 737 edges, 43 node features
019688.pdf (corel pdf engine): 62 feature values | graph 216 nodes, 538 edges, 43 node features

CELL 21 ready. Expected: 62 feature values (61 if a file has no page streams), 43 node features.


In [4]:
# CELL 22 — Full build runner: per chunk -> generate 15 versions -> features -> graphs -> QA -> save -> delete PDFs; then merge all finished chunks
import os, glob, gzip, pickle, random, time
import numpy as np
import pandas as pd
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

RUN_CHUNKS = list(range(len(CHUNK_LIST)))  # TEST RUN: chunk 0 only. For the full run change to: RUN_CHUNKS = list(range(len(CHUNK_LIST)))

FULL_FEATS = f"{WORK}/full_features.csv"
FULL_GRAPHS = f"{WORK}/full_graphs.pkl.gz"
FULL_FAILS = f"{WORK}/full_failures.csv"
QA_PAIRS = [("F1_digit_resave", "G1_resave"), ("F2_digit_recompress", "G2_recompress"), ("F3_digit_incr", "G3_note_incr"),
            ("F4_value_resave", "G1_resave"), ("F5_cover_replace", "G1_resave"), ("F6_digit_linearize", "G4_linearize")]

def cpaths(ci):
    b = f"{CHUNKS}/chunk_{ci:02d}"
    return f"{b}_features.csv", f"{b}_graphs.pkl.gz", f"{b}_failures.csv"

def disk_gb(p="/kaggle/working"):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs) / 1e9

def run_pool(fn, items, timeout, desc):
    results, errors = {}, {}
    with ProcessPool(max_workers=WORKERS) as pool_ex:
        futs = {pool_ex.schedule(fn, args=(it,), timeout=timeout): it for it in items}
        for fu in tqdm(as_completed(futs), total=len(futs), desc=desc):
            it = futs[fu]
            try:
                results[it] = fu.result()
            except TimeoutError:
                errors[it] = "Timeout"
            except Exception as ex:
                errors[it] = f"{type(ex).__name__}: {str(ex)[:60]}"
    return results, errors

def page_text(path, i):
    with fitz.open(path) as d:
        return d[int(i)].get_text() if int(i) < len(d) else ""

def save_atomic_csv(df, path):
    df.to_csv(path + ".tmp", index=False)
    os.replace(path + ".tmp", path)

def run_chunk(ci):
    fcsv, fgr, ffail = cpaths(ci)
    if os.path.exists(fcsv) and os.path.exists(fgr):
        print(f"Chunk {ci}: already done, skipping")
        return
    jobs = CHUNK_LIST[ci]
    t0 = time.time()
    fams = pd.Series([f for _, f in jobs]).value_counts().to_dict()
    print(f"\n=== CHUNK {ci} / {len(CHUNK_LIST) - 1}: {len(jobs)} originals {fams} ===")

    # ---------- 1/3 generate ----------
    gen, gerr = run_pool(process, jobs, TIME_LIMIT, f"Chunk {ci} | 1/3 generate")
    rows, fails = [], []
    for job in jobs:
        res = gen.get(job)
        if res is not None and res[1] is not None:
            rows += res[1]
        else:
            reason = gerr.get(job) or (res[2] if res is not None else "missing")
            fails.append({"orig_file": job[0], "family": job[1], "stage": "generate", "reason": reason})
            cleanup(job[0][:-4])
    man = pd.DataFrame(rows)
    if man.empty:
        print("No documents generated in this chunk")
        return
    paths = [os.path.join(OUT, f) for f in man.out_file]

    # ---------- 2/3 features, 3/3 graphs ----------
    fres, ferr = run_pool(extract_features, paths, FEAT_TIME_LIMIT, f"Chunk {ci} | 2/3 features")
    gres, grerr = run_pool(to_graph, paths, GRAPH_TIME_LIMIT, f"Chunk {ci} | 3/3 graphs")

    # ---------- drop any document whose 15 files are not all complete (keeps twins together) ----------
    orig_of = dict(zip(man.out_file, man.orig_file))
    bad = {}
    for stage, errs in [("features", ferr), ("graph", grerr)]:
        for p, e in errs.items():
            bad.setdefault(orig_of[os.path.basename(p)], (stage, e))
    for o, (stage, e) in bad.items():
        fails.append({"orig_file": o, "family": man.loc[man.orig_file == o, "family"].iloc[0], "stage": stage, "reason": e})
    ok = man[~man.orig_file.isin(bad)].reset_index(drop=True)

    # ---------- quick QA: does the fake page text really differ from its honest twin? ----------
    qa = {f: [0, 0] for f, _ in QA_PAIRS}
    docs = sorted(ok.orig_file.unique())
    for o in random.Random(ci).sample(docs, min(5, len(docs))):
        sub = ok[ok.orig_file == o].set_index("variant")
        for f, g in QA_PAIRS:
            try:
                changed = page_text(os.path.join(OUT, sub.loc[f, "out_file"]), sub.loc[f, "edit_page"]) != \
                          page_text(os.path.join(OUT, sub.loc[g, "out_file"]), sub.loc[f, "edit_page"])
                qa[f][0] += int(changed)
                qa[f][1] += 1
            except Exception:
                pass

    # ---------- save (graphs + failures first, features CSV last = "chunk done" marker) ----------
    feats = ok.merge(pd.DataFrame([fres[p] for p in paths if orig_of[os.path.basename(p)] not in bad]),
                     on="out_file", how="left")
    meta_cols = ["out_file", "orig_file", "family", "variant", "label", "tool", "save_mode", "edit_type",
                 "edit_page", "twin", "split"]
    meta = ok[meta_cols].set_index("out_file")
    graphs = []
    for f in ok.out_file:
        g = gres[os.path.join(OUT, f)]
        g.update(meta.loc[f].to_dict())
        g["out_file"] = f
        graphs.append(g)
    with gzip.open(fgr + ".tmp", "wb") as fh:
        pickle.dump({"graphs": graphs, "feat_names": FEAT_NAMES, "node_types": NODE_TYPES, "edge_keys": EDGE_KEYS},
                    fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(fgr + ".tmp", fgr)
    save_atomic_csv(pd.DataFrame(fails, columns=["orig_file", "family", "stage", "reason"]), ffail)
    save_atomic_csv(feats, fcsv)

    # ---------- delete this chunk's PDFs ----------
    for p in glob.glob(os.path.join(OUT, "*.pdf")):
        os.remove(p)

    print(f"Chunk {ci} done in {(time.time() - t0) / 60:.1f} min | documents kept: {ok.orig_file.nunique()} of {len(jobs)}"
          f" | files: {len(ok)} | failed docs: {len(fails)}")
    if fails:
        print("Failure reasons:", pd.Series([f"{x['stage']}: {x['reason'].split(':')[0]}" for x in fails]).value_counts().to_dict())
    print("QA (fake page text differs from twin):", {f.split("_")[0]: f"{c}/{n}" for f, (c, n) in qa.items()})
    print(f"Disk used in /kaggle/working: {disk_gb():.2f} GB")

# ---------- run ----------
print("Chunks to run now:", RUN_CHUNKS, "| total chunks:", len(CHUNK_LIST))
for ci in RUN_CHUNKS:
    run_chunk(ci)

# ---------- merge every finished chunk into one features file + one graphs file ----------
done = sorted(glob.glob(f"{CHUNKS}/chunk_*_features.csv"))
print(f"\n=== MERGING {len(done)} finished chunk(s) of {len(CHUNK_LIST)} ===")
feats_all = pd.concat([pd.read_csv(p) for p in done], ignore_index=True)
fails_all = pd.concat([pd.read_csv(p.replace("_features.csv", "_failures.csv")) for p in done], ignore_index=True)
graphs_all = []
for p in tqdm(done, desc="Merging graphs"):
    with gzip.open(p.replace("_features.csv", "_graphs.pkl.gz"), "rb") as fh:
        graphs_all += pickle.load(fh)["graphs"]
assert len(graphs_all) == len(feats_all), "graphs and features do not match"
save_atomic_csv(feats_all, FULL_FEATS)
save_atomic_csv(fails_all, FULL_FAILS)
with gzip.open(FULL_GRAPHS + ".tmp", "wb") as fh:
    pickle.dump({"graphs": graphs_all, "feat_names": FEAT_NAMES, "node_types": NODE_TYPES, "edge_keys": EDGE_KEYS},
                fh, protocol=pickle.HIGHEST_PROTOCOL)
os.replace(FULL_GRAPHS + ".tmp", FULL_GRAPHS)

docs = feats_all.drop_duplicates("orig_file")
print("Files:", len(feats_all), "| documents:", len(docs), "| graphs file:", round(os.path.getsize(FULL_GRAPHS) / 1e6, 1), "MB")
print("\nDocuments per family:")
print(docs.family.value_counts().reindex(FAMILIES).fillna(0).astype(int).to_string())
print("\nDocuments per split:", docs.split.value_counts().to_dict())
print("\nFiles per version (should all be equal):")
print(feats_all.variant.value_counts().sort_index().to_string())
print("\nAll failures so far:", len(fails_all))
if len(fails_all):
    print(fails_all.assign(r=fails_all.stage + ": " + fails_all.reason.str.split(":").str[0]).r.value_counts().head(8).to_string())
print(f"\nDisk used in /kaggle/working: {disk_gb():.2f} GB")

Chunks to run now: [0] | total chunks: 10

=== CHUNK 0 / 9: 300 originals {'acrobat distiller windows': 300} ===


Chunk 0 | 1/3 generate:   0%|          | 0/300 [00:00<?, ?it/s]

Chunk 0 | 2/3 features:   0%|          | 0/4365 [00:00<?, ?it/s]

Chunk 0 | 3/3 graphs:   0%|          | 0/4365 [00:00<?, ?it/s]

Chunk 0 done in 15.2 min | documents kept: 291 of 300 | files: 4365 | failed docs: 9
Failure reasons: {'generate: RuntimeError': 7, 'generate: Timeout': 2}
QA (fake page text differs from twin): {'F1': '5/5', 'F2': '5/5', 'F3': '5/5', 'F4': '5/5', 'F5': '5/5', 'F6': '5/5'}
Disk used in /kaggle/working: 0.02 GB

=== MERGING 1 finished chunk(s) of 10 ===


Merging graphs:   0%|          | 0/1 [00:00<?, ?it/s]

Files: 4365 | documents: 291 | graphs file: 20.8 MB

Documents per family:
family
acrobat distiller windows    291
acrobat distiller for          0
acrobat pdfwriter for          0
adobe pdf library              0
corel pdf engine               0

Documents per split: {'train': 244, 'dev': 47}

Files per version (should all be equal):
variant
F1_digit_resave        291
F2_digit_recompress    291
F3_digit_incr          291
F4_value_resave        291
F5_cover_replace       291
F6_digit_linearize     291
F7_stamp_fake          291
F8_textbox_fake        291
G0_original            291
G1_resave              291
G2_recompress          291
G3_note_incr           291
G4_linearize           291
G5_stamp_honest        291
G6_textbox_honest      291

All failures so far: 9
r
generate: RuntimeError    7
generate: Timeout         2

Disk used in /kaggle/working: 0.05 GB
